In [ ]:
# =============================================================================
# T_Dubber native pack -- extract once, pin absolute paths, verify strictly
# =============================================================================
# The pack dataset mounts read-only on Kaggle, and the current upload shape is
# a .zip wrapping kaggle-tdubber-pack.tar.gz (Kaggle does not auto-extract a
# tar.gz). This cell lands the tree on a writable filesystem
# (/kaggle/working/tdubber_pack), restores the execute bits, prepends its
# bin/ to PATH, and pins every tool by absolute path.
#
# PINNING, not PATH alone: Kaggle's base image ships a foreign `normalizer`
# (a Python argparse tool with a `-t THRESHOLD` CLI). PATH-only setups picked
# it up, it exited 2 on our four-argument call, and the run silently dropped
# to the slow Python engine. The env overrides below are checked before any
# PATH lookup inside mazinger.assemble, and the PATH prepend is verified
# afterwards.
#
# Never set MAZINGER_NORMALIZER=off in this notebook: that would silently
# disable the pre-compiled engine the pack exists to provide.

import os
import shutil
import subprocess
import tarfile
import zipfile
from pathlib import Path

PACK_WORKDIR = Path("/kaggle/working/tdubber_pack")
PACK_BIN_DIR = PACK_WORKDIR / "bin"
PACK_TOOLS = ("tgup", "edge-fetch", "stitcher", "normalizer", "subtitle_forge", "havaldar_core")
PACK_TOOL_ENV = {
    "tgup": "TGUP_BIN",
    "edge-fetch": "EDGE_FETCH_BIN",
    "stitcher": "STITCHER_BIN",
    "normalizer": "NORMALIZER_BIN",
    "subtitle_forge": "SUBTITLE_FORGE_BIN",
}


def _find_pack_source():
    # 1) An already-extracted tree (bin/ + MANIFEST.txt) needs no unpacking.
    for manifest in Path("/kaggle/input").rglob("MANIFEST.txt"):
        if (manifest.parent / "bin").is_dir():
            return manifest.parent
    # 2) A bare archive, whichever wrapper this upload used.
    for pattern in (
        "kaggle-tdubber-pack.tar.gz",
        "kaggle-tdubber-pack.zip",
        "*tdubber*pack*.tar.gz",
        "*tdubber*pack*.zip",
    ):
        for hit in sorted(Path("/kaggle/input").rglob(pattern)):
            if hit.is_file():
                return hit
    return None


def _unpack_pack(source, workdir):
    # A .zip may wrap the real tar.gz (the current upload does); unwrap one
    # level, then extract the tar. A zip can also BE the extracted tree.
    if source.suffix.lower() == ".zip":
        with zipfile.ZipFile(source) as bundle:
            inner_tar = next(
                (name for name in bundle.namelist() if name.endswith(".tar.gz")), None
            )
            if inner_tar is not None:
                bundle.extract(inner_tar, workdir)
                source = workdir / inner_tar
            else:
                bundle.extractall(workdir)
                return (workdir / "bin").is_dir()
    if tarfile.is_tarfile(source):
        with tarfile.open(source) as bundle:
            bundle.extractall(workdir, filter="data")
    return (workdir / "bin").is_dir()


def _normalizer_cli_ok(binary):
    # Our C++ CLI prints "<input_wav> <output_wav> ..." and exits 2 with no
    # arguments; any other tool resolved under this name fails this probe.
    try:
        probe = subprocess.run(
            [str(binary)], capture_output=True, text=True, timeout=15
        )
    except Exception as exc:
        return False, "could not execute: %s" % exc
    output = ((probe.stdout or "") + "\n" + (probe.stderr or "")).strip()
    first_line = output.splitlines()[0] if output else ""
    return "input_wav" in output, first_line


pack_source = _find_pack_source()
if pack_source is None:
    print(
        "WARNING: T_Dubber pack not found under /kaggle/input. "
        "The pipeline will use the slow Python normalizer. "
        "Attach the kaggle-tdubber-pack dataset to this kernel.",
        flush=True,
    )
else:
    if PACK_WORKDIR.exists():
        shutil.rmtree(PACK_WORKDIR)
    PACK_WORKDIR.mkdir(parents=True, exist_ok=True)
    if pack_source.is_dir():
        shutil.copytree(pack_source, PACK_WORKDIR, dirs_exist_ok=True)
        unpacked = PACK_BIN_DIR.is_dir()
    else:
        print("Unpacking %s -> %s" % (pack_source, PACK_WORKDIR), flush=True)
        unpacked = _unpack_pack(pack_source, PACK_WORKDIR)
    if not unpacked:
        raise RuntimeError(
            "T_Dubber pack was found at %s but produced no bin/ directory; "
            "refusing to continue with a broken native toolkit." % pack_source
        )

    # Execute bits survive neither a zip nor a read-only mount; set them here.
    for entry in PACK_BIN_DIR.iterdir():
        if entry.is_file():
            entry.chmod(0o755)

    # Absolute pins first (checked before PATH inside mazinger.assemble),
    # then the PATH prepend for tools that are resolved by name elsewhere.
    for tool in PACK_TOOLS:
        tool_path = PACK_BIN_DIR / tool
        env_var = PACK_TOOL_ENV.get(tool)
        if tool_path.is_file() and env_var:
            os.environ[env_var] = str(tool_path)
    os.environ["PATH"] = str(PACK_BIN_DIR) + os.pathsep + os.environ.get("PATH", "")

    present = ", ".join(sorted(p.name for p in PACK_BIN_DIR.iterdir() if p.is_file()))
    print("Pack unpacked: %s" % present, flush=True)
    print("NORMALIZER_BIN -> %s" % os.environ.get("NORMALIZER_BIN"), flush=True)
    print("STITCHER_BIN   -> %s" % os.environ.get("STITCHER_BIN"), flush=True)
    print("TGUP_BIN       -> %s" % os.environ.get("TGUP_BIN"), flush=True)

    # Strict enforcement: the pinned C++ normalizer is the ONLY acceptable
    # resolution. If the pack shipped a broken binary, or the PATH prepend
    # lost to a system tool, fail NOW -- a whole GPU run silently on the
    # Python fallback is the bug this cell exists to kill.
    normalizer = PACK_BIN_DIR / "normalizer"
    ok, first_line = _normalizer_cli_ok(normalizer)
    if not ok:
        raise RuntimeError(
            "Pack normalizer failed its CLI identity check (%s); refusing to "
            "silently run on the Python fallback. First line: %r"
            % (normalizer, first_line)
        )
    resolved = shutil.which("normalizer")
    print("shutil.which('normalizer') -> %s" % resolved, flush=True)
    if resolved != str(normalizer):
        raise RuntimeError(
            "PATH still resolves 'normalizer' to %r instead of the pack's %r. "
            "mazinger.assemble will still prefer NORMALIZER_BIN, but this "
            "environment is not the one the pack was built for."
            % (resolved, str(normalizer))
        )
    print("C++ normalizer ready: %s" % normalizer, flush=True)

In [ ]:
# =============================================================================
# T_Dubber edge cache -- fetch warm artefacts before anything is installed
# =============================================================================
# WHAT THIS CELL IS FOR
# ---------------------
# Run 14 took 1220 s. 903 s of that -- 74% -- was downloading bytes that had not
# changed since the previous run:
#
#     pip install of pylibs         378 s
#     Homura-2B snapshot download   525 s
#
# The GPU work (TTS 221 s, transcribe 40 s) has to stay on Kaggle, because there
# is no GPU anywhere else in the free tier. The downloads do not. This cell pulls
# prebuilt artefacts from the origin -- a public Hugging Face repository, served
# from /resolve/main/ -- content-addressed by sha256, and the next cell's
# find_pylibs() finds them already in place.
#
# WHY THE ORIGIN IS A REPOSITORY AND NOT A SPACE
# ----------------------------------------------
# A Docker Space on Hugging Face costs $9/month: since 2026, "Gradio and Docker
# Spaces run on compute and require a paid plan to create". A public dataset
# repository is free, has no sleep state, no cold start and no volume that is
# wiped on redeploy -- and this cell only ever GETs bytes, so it needs no
# process to talk to. The repository IS the server.
#
# WHAT IT IS NOT
# --------------
# It is not required. If EDGE_URL is unset, or the origin is unreachable, or the
# manifest does not cover what this run needs, this cell prints one line and
# exits. Everything downstream is unchanged. A cache miss must never be worse
# than no cache.
#
# WHY THIS RUNS BEFORE THE INSTALL
# -------------------------------
# Ordering is the entire value. Run after the pip install it would have nothing
# left to save; run before, it removes the install. The cell sits directly after
# the pack cell (which provides the binary) and before the dependency cell.
#
# WHY THIS IS NOT A WARM CACHE, AND WHY THAT MATTERS
# --------------------------------------------------
# An earlier version of this cell fetched into /kaggle/working and advertised a
# 33 ms warm path. That number is unreachable on Kaggle and the cell no longer
# claims it:
#
#   * a NEW Kaggle session starts with an EMPTY /kaggle/working, so nothing this
#     cell writes survives to the next run -- the digest comparison is right but
#     always answers "missing";
#   * /kaggle/working is also cleared whenever the session ends, so the cache
#     cannot outlive the run that built it even in principle;
#   * the durable, free storage Kaggle offers is an attached DATASET under
#     /kaggle/input, and the next cell already looks there first
#     (find_pylibs([INPUT, WORK])).
#
# So this cell is a COLD-RUN accelerator, not a cache. What it actually saves is
# the pip resolver and the per-package install: one tar extract instead of ~110
# wheel installations. It cannot save the download, because the bytes are the
# same either way.
#
# The genuinely free win on repeat runs is attaching the built tree as a Kaggle
# Dataset, which makes it local disk and costs zero seconds to "download". That
# is already implemented downstream; this cell is what makes the FIRST run cheap
# enough to be worth keeping.
#
# WHY THE DIGEST MATTERS
# ----------------------
# pylibs is unpacked into sys.path and then imports and executes code. An archive
# that arrived truncated or substituted is not a failed run, it is arbitrary code
# execution. edge-fetch therefore verifies sha256 against the Space's manifest
# and installs nothing on a mismatch, and writes to a temporary name so a partial
# file is never visible under the real path.

import os
import subprocess
import sys
import time
from pathlib import Path

T_EDGE = time.monotonic()

WORK = Path("/kaggle/working")

# The origin. A Kaggle notebook cannot be handed environment variables by the
# kernel metadata -- there is no field for it -- so the default lives here
# rather than being injected, and EDGE_URL still overrides it for a local run,
# a mirror, or a LAN server during development.
#
# The trailing /resolve/main/ is what makes a plain public Hugging Face
# repository behave like a file server: the client appends /manifest.json and
# /artifact/<name>, and both resolve to real file paths in the repository.
# Point EDGE_URL at anything else with that layout and this cell works
# unchanged -- a Space, an nginx, a laptop on the same network.
DEFAULT_EDGE_URL = "https://huggingface.co/datasets/pocotarun/tdubber-edge/resolve/main"
EDGE_URL = (os.environ.get("EDGE_URL", "") or DEFAULT_EDGE_URL).strip()

# Roles to pull. "pack" is not among them: the native pack already arrives via
# the Input dataset, which is faster than any HTTP transfer for 7 small files,
# and unpacking a tar of binaries would cost more than it saves.
#
# "weights" is not among them either. huggingface_hub already fetches the model
# from HF's own CDN, and a second copy of 5 GB served from a Space cannot beat
# that; it can only add a sleep/wake dependency. Only "pylibs" is worth
# replacing, because that is the one artefact pip has to resolve and install
# package by package.
EDGE_ROLES = ("pylibs",)
EDGE_DEST = WORK / "edge_cache"


def _edge_tick(label):
    print("[%6.1fs] edge: %s" % (time.monotonic() - T_EDGE, label), flush=True)


def _find_edge_fetch():
    """Locate the edge-fetch binary the pack provides.

    Same pin-by-absolute-path discipline as the normalizer in the pack cell: a
    PATH-only lookup would resolve whatever the Kaggle base image happens to
    call "edge-fetch", which is the failure mode that once made this pipeline
    silently run a foreign normalizer.
    """
    pinned = os.environ.get("EDGE_FETCH_BIN", "").strip()
    candidates = []
    if pinned:
        candidates.append(Path(pinned))
    bin_dir = WORK / "tdubber_pack" / "bin"
    candidates.append(bin_dir / "edge_fetch")
    candidates.append(bin_dir / "edge-fetch")
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    return None


def main():
    if not EDGE_URL:
        print("edge: EDGE_URL is not set; skipping the edge cache", flush=True)
        return
    if not EDGE_URL.startswith(("http://", "https://")):
        # Refusing a bare host here keeps the mistake visible in the log instead
        # of surfacing as an opaque dial error from inside the Go client.
        print("edge: EDGE_URL %r is not an http(s) URL; skipping" % EDGE_URL, flush=True)
        return

    binary = _find_edge_fetch()
    if binary is None:
        print(
            "edge: edge_fetch binary not found in the pack; skipping "
            "(the dependency cell will install from origin)",
            flush=True,
        )
        return

    try:
        binary.chmod(0o755)
    except OSError:
        pass

    EDGE_DEST.mkdir(parents=True, exist_ok=True)

    # One invocation per role. A role the Space has not published exits 0 without
    # fetching, so this needs no "is it published" pre-check -- the manifest is
    # the authority and asking it twice would be a second source of truth.
    for role in EDGE_ROLES:
        started = time.monotonic()
        _edge_tick("fetching role %s from %s" % (role, EDGE_URL))
        # -retries 3 rather than the default 4: this cell sits inside a notebook
        # that has its own total budget, and spinning here delays the fallback to
        # origin that actually makes progress.
        result = subprocess.run(
            [
                str(binary),
                "-url", EDGE_URL,
                "-dest", str(EDGE_DEST),
                # Archives are rooted at /kaggle/working: pylibs.tar.gz carries
                # pylibs/ and the weights archive carries hf_cache/, so one
                # extraction directory serves both roles and lands each tree
                # exactly where the next cell and HF_HOME look for it.
                "-unpack-dir", str(WORK),
                "-role", role,
                "-retries", "3",
            ],
            capture_output=True,
            text=True,
            timeout=30 * 60,
            check=False,
        )
        elapsed = time.monotonic() - started
        output = ((result.stdout or "") + (result.stderr or "")).strip()

        for line in output.splitlines():
            print("    edge: %s" % line, flush=True)

        if result.returncode != 0:
            # Non-zero here means -must-fetch semantics were somehow in force.
            # The cell's contract is that a cache failure is never fatal, so the
            # run continues on the origin path.
            print(
                "edge: role %s returned %d; continuing on the origin path"
                % (role, result.returncode),
                flush=True,
            )
        else:
            _edge_tick("role %s settled in %.1fs" % (role, elapsed))

    # Report what actually landed, and -- the part that was wrong before -- say
    # plainly that a second run will NOT be faster because of this cell. Without
    # that line "nothing was downloaded and it was still slow" is
    # indistinguishable from "the cache did not engage", which is how a
    # non-functional cache gets left in place for months.
    # The interesting number is no longer what sits in edge_cache: archives are
    # unpacked and then DELETED to stay under the 20 GB output quota, so an
    # empty role directory there means success rather than a miss. What matters
    # is the tree the next cell consumes, so that is what gets reported -- along
    # with the ready marker, because find_pylibs() refuses a tree without it and
    # a missing marker silently falls back to the 458 s pip install.
    trees = {
        "pylibs": (WORK / "pylibs", WORK / "pylibs" / ".tdubber_ready"),
        "weights": (WORK / "hf_cache", None),
    }
    for role in EDGE_ROLES:
        tree, marker = trees.get(role, (None, None))
        if tree is None:
            continue
        if not tree.is_dir():
            _edge_tick("role %s: no tree at %s (origin path will be used)" % (role, tree))
            continue
        files = [p for p in tree.rglob("*") if p.is_file()]
        total = sum(p.stat().st_size for p in files)
        state = ""
        if marker is not None:
            state = ", ready_marker=%s" % ("yes" if marker.is_file() else "MISSING")
        _edge_tick(
            "role %s: tree at %s (%d file(s), %.1f MiB%s)"
            % (role, tree, len(files), total / (1 << 20), state)
        )
    _edge_tick(
        "note: /kaggle/working is empty on every NEW session, so this is a "
        "cold-run saving only. For a warm run, attach the built tree as a "
        "Kaggle Dataset (the next cell checks /kaggle/input first)."
    )


main()


In [ ]:
# Install Mazinger and start a local, OpenAI-compatible Homura server.
#
# WHERE THE TIME USED TO GO (Version 14 run, seconds from kernel start):
#     0 ->  48   pip install of the mazinger/speech deps
#    49 -> 379   pip install vllm==0.29.0  -- ~100 s downloading, then ~215 s of
#                                    "Attempting uninstall": pip was ripping the
#                                    image's torch 2.10.0+cu128, torchaudio, triton
#                                    and setuptools out of site-packages before
#                                    replacing them with torch 2.13.0 + a CUDA 13
#                                    wheel set (nvidia-*-13.x).
#   391 -> 916   Homura-2B download + server start (nothing was cached)
#   ------------------------------------------------------------- 916 s ~ 15 min
#
# This cell attacks exactly those three numbers:
#   * deps go into /kaggle/working/pylibs via `pip install --target`, which NEVER
#     touches site-packages -- so there is nothing to uninstall, and a warm run
#     skips pip completely (0 s instead of 378 s);
#   * HF_HOME moves out of /root/.cache (dies with the container) into
#     /kaggle/working/hf_cache, which Kaggle saves with the notebook output;
#   * both are also searched under /kaggle/input first, because a NEW session
#     starts with an EMPTY /kaggle/working: Kaggle saves output (20 GB quota) but
#     does not restore it automatically -- this notebook's output, or a cache
#     dataset, has to be attached as Input. Mounted cache = warm run, nothing
#     mounted = cold run, and the log always says which one happened;
#   * every phase prints its own elapsed time, so the 1-2 min target stays
#     measurable instead of guessed.
import os
import sys
import time
import json
import shutil
import zipfile
import subprocess
import urllib.request
import signal
from pathlib import Path

T0 = time.monotonic()


def _tick(label):
    print(f"[{time.monotonic() - T0:6.1f}s] {label}", flush=True)


WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
WORK.mkdir(parents=True, exist_ok=True)

# --- Task 1: keep the model weights out of /root/.cache ----------------------
# Has to run before anything imports huggingface_hub, which is why it is here at
# the top of the cell and not next to the vLLM launch.
HF_CACHE = WORK / "hf_cache"
os.environ.setdefault("HF_HOME", str(HF_CACHE))
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")  # xet uploads stall on Kaggle
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
_tick(f"HF_HOME={os.environ['HF_HOME']}")

os.environ["OPENAI_API_KEY"] = "EMPTY"
os.environ["OPENAI_BASE_URL"] = "http://localhost:8000/v1"
os.environ["OPENAI_MODEL"] = "IndexTeam/Index-Homura-2B"
os.environ["MAZINGER_DISABLE_VISION"] = "1"
os.environ["MAZINGER_LLM_MAX_OUTPUT_TOKENS"] = "2048"

MODEL_ID = os.environ["OPENAI_MODEL"]
READY_MARKER = ".tdubber_ready"
PYLIBS_DIR = WORK / "pylibs"

SHALLOW_PROBE = """
import importlib.util as u, sys
names = ["vllm", "torch", "torchaudio", "torchvision", "faster_whisper", "av",
          "openai", "demucs", "omnivoice"]
missing = [n for n in names if u.find_spec(n) is None]
if missing:
    sys.exit("missing: " + ", ".join(missing))
print("deps present: " + ", ".join(names))
"""
# Only names whose import name is certainly the distribution name, because this
# probe failing means a full reinstall.
#
# The deep probe must go past `import vllm`. That import does not touch
# torchvision, so a tree with a mismatched torch/torchvision pair passes it
# and prints "deps ok" -- and then `vllm serve` dies importing
# transformers -> torchvision with "operator torchvision::nms does not
# exist". Importing torchvision and then calling the op is what proves the
# C++ extension is actually bound to this torch build.
DEEP_PROBE = """
import vllm, torch, faster_whisper, av, openai
import torchvision
import torchaudio
import vllm.entrypoints.cli.main
torch.ops.torchvision.nms
# Importing torchaudio IS the alignment check: the function that raised in the
# failed run is torchaudio's own _check_cuda_version(). Probing it here costs a
# second and names the cause, instead of surfacing it 20 minutes later inside the
# server start, by which point the HF publish cell never runs.
print("deps ok: vllm %s / torch %s (cuda %s) / torchaudio %s / torchvision %s" % (
    vllm.__version__, torch.__version__, torch.version.cuda,
    torchaudio.__version__, torchvision.__version__))
"""


def probe_deps(code, pylibs):
    """Run a probe in a subprocess that sees `pylibs`, without mutating this kernel.

    Probing in a child (rather than importing here) keeps the check side-effect
    free: a failed probe can still switch the whole cell over to the system
    install path, which would be impossible if sys.path had already been edited.
    """
    env = os.environ.copy()
    if pylibs:
        env["PYTHONPATH"] = os.pathsep.join(
            part for part in [str(pylibs), env.get("PYTHONPATH", "")] if part
        )
    return subprocess.run([sys.executable, "-c", code], env=env).returncode


def find_model_copy(bases):
    """A local snapshot of Homura-2B, mounted or saved from a previous run.

    Handing `vllm serve` a directory downloads nothing and writes nothing --
    which is why this beats HF_HOME: /kaggle/input is read-only, so a cache
    living there could not be updated even if we wanted to.
    """
    key = "models--" + MODEL_ID.replace("/", "--")
    for base in bases:
        if not base:
            continue
        try:
            for snap in Path(base).glob(f"**/{key}/snapshots/*"):
                if (snap / "config.json").exists() and any(snap.glob("*.safetensors")):
                    return snap
        except OSError:
            continue
    return None


def find_pylibs(bases):
    """A previously installed dependency tree, mounted or saved from a last run.

    The ready marker is written only after the install AND its import probe both
    passed, so a half-finished install is never mistaken for a warm cache.
    """
    for base in bases:
        if not base:
            continue
        try:
            for candidate in Path(base).glob("**/pylibs"):
                if (candidate / READY_MARKER).exists() and (candidate / "vllm" / "__init__.py").exists():
                    return candidate
        except OSError:
            continue
    return None


# --- mazinger source ---------------------------------------------------------
archive_candidates = list(INPUT.rglob("mazinger_source.zip"))
mazinger_extract_path = WORK / "mazinger"
search_root = INPUT
if archive_candidates:
    if mazinger_extract_path.exists():
        shutil.rmtree(mazinger_extract_path)
    mazinger_extract_path.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_candidates[0]) as source_archive:
        extraction_root = mazinger_extract_path.resolve()
        for member in source_archive.infolist():
            destination = (extraction_root / member.filename).resolve()
            if destination != extraction_root and extraction_root not in destination.parents:
                raise RuntimeError("Mazinger source archive contains an unsafe path.")
        source_archive.extractall(extraction_root)
    search_root = mazinger_extract_path
else:
    print("Kaggle mounted the source archive as an extracted folder; locating pyproject.toml directly.", flush=True)
mazinger_candidates = [
    path.parent for path in search_root.rglob("pyproject.toml")
    if (path.parent / "mazinger").is_dir()
    or (path.parent / "src" / "mazinger").is_dir()
    or path.parent.name.lower() in {"mazinger", "mazinger_source"}
]
if not mazinger_candidates:
    mounted_files = [str(path) for path in INPUT.rglob("pyproject.toml")]
    raise FileNotFoundError(f"Mazinger pyproject.toml not found in Kaggle input. Found: {mounted_files}")
mazinger_path = str(mazinger_candidates[0])
# Import the bundled source directly (no wheel build needed).
os.environ["PYTHONPATH"] = mazinger_path + os.pathsep + os.environ.get("PYTHONPATH", "")

# --- dependencies ------------------------------------------------------------
# `pip --target` resolves against an EMPTY directory -- it happily re-downloads
# what the image already has -- but in exchange it never runs a single
# "Attempting uninstall". That one difference is where ~215 s of the 330 s
# install went, and it is also why the result is a self-contained, relocatable
# tree that can be mounted read-only as Input next time.
# Installed last and as one unit, so every wheel of the family carries the
# same CUDA tag. `pip --target` resolves against an empty directory, which
# is why this has to be forced rather than merely requested.
TORCH_TRIPLE = [
    "torch==2.11.0+cu128", "torchvision==0.26.0+cu128", "torchaudio==2.11.0+cu128",
]

SPEECH_DEPS = [
    "yt-dlp>=2026.3.17", "openai>=1.0", "json-repair>=0.28", "Pillow>=10.0",
    "soundfile>=0.12", "numpy>=1.24", "tqdm>=4.60", "python-slugify>=8.0",
    "faster-whisper", "av>=14.0.0", "demucs", "omnivoice",
]


def run_pip(packages, label):
    cmd = [sys.executable, "-m", "pip", "install", "--no-cache-dir",
           "--target", str(PYLIBS_DIR)] + packages
    _tick(f"pip [{label}]: {' '.join(packages[:2])}{' ...' if len(packages) > 2 else ''}")
    return subprocess.run(cmd).returncode


def install_deps():
    """Fill pylibs; True when the tree landed.

    Two passes, and the split point is the whole point:

      1. vLLM, which hard-pins the torch family -- ONE resolver pass, so the
         three agree. vLLM 0.26.0 pins torch==2.11.0 / torchvision==0.26.0 /
         torchaudio==2.11.0, all three of which the cu128 index actually carries.
         vLLM 0.27.0+ pins torch==2.13.0, and the cu128 index has no 2.13.0 at
         all, so that pair of requests cannot both be satisfied.
      2. speech/mazinger deps.

    Asking for that torch family across separate passes is what broke this.
    `pip --target` resolves against an EMPTY directory, so a later pass picks
    its own newest torch, then refuses to overwrite the files the earlier pass
    already wrote ("Target directory ... already exists. Specify --upgrade").
    450 such warnings, and the tree ends up holding one pass's torch next to
    the other pass's torchvision. vLLM then dies on import with
    "operator torchvision::nms does not exist" -- a torch whose C++ ops the
    installed torchvision was not built against.

    Pass 2 is safe exactly because pass 1 landed first: every speech dep that
    needs torch accepts the version vLLM pinned, so pip resolves nothing new
    for torch and leaves the installed tree alone.
    """
    PYLIBS_DIR.mkdir(parents=True, exist_ok=True)

    # One index, one CUDA build, one atomic install of the torch family.
    #
    # `--extra-index-url` does not prefer an index -- pip takes the best version
    # match across both. That is how the failed run ended up with torch 2.13.0
    # (PyPI = CUDA 13.0) beside torchaudio 2.11.0+cu128 (cu128 index): two CUDA
    # stacks from a single resolver pass, and a crash 20 minutes later inside
    # `import torchaudio`. vLLM 0.26.0 is the newest release whose three pins
    # all exist on the cu128 index; 0.27.0+ moved to torch 2.13.0, which the
    # cu128 index does not carry at all.
    cu128 = ["--extra-index-url", "https://download.pytorch.org/whl/cu128"]
    triple = ["--index-url", "https://download.pytorch.org/whl/cu128", "--force-reinstall", "--no-deps"]
    if run_pip(["vllm==0.26.0"] + cu128, "vllm+cu128") != 0:
        _tick("pip: vllm cu128 index failed; retrying from plain PyPI")
        if run_pip(["vllm==0.26.0"], "vllm plain") != 0:
            return False
    # Force the exact family in afterwards. `--no-deps` stops pip from pulling a
    # fourth opinion about torch; `--index-url` (not --extra) means there is no
    # other place a cu13 wheel could come from.
    if run_pip(TORCH_TRIPLE + triple, "torch triple (cu128)") != 0:
        print("WARNING: could not force the cu128 torch family; the probe decides.", flush=True)

    # Pass 2 gets the SAME extra-index AND an explicit torch pin.
    #
    # `pip --target` resolves against an EMPTY directory: it cannot see the tree
    # pass 1 just wrote. So the unpinned form of this line resolved torch to
    # 2.14.1 (verified with a dry run against live PyPI), downloaded that wheel
    # -- ~554 MB -- and then refused to overwrite the 2.13.0 files vLLM had
    # pinned, emitting the "Target directory ... already exists" churn and
    # leaving the right version in place anyway. Correct result, wasted download,
    # every single cold run.
    #
    # Pinning to the version installed above makes the second resolve agree with
    # the first, so the wheel is fetched once instead of twice. The extra-index has
    # to match too: without it pip looks at PyPI, where the torch CUDA wheels
    # are named `nvidia-*-cu13`, while the cu128 index serves `nvidia-*-cu12`.
    # That naming difference is what makes the unpinned pass drag in a second,
    # different CUDA stack.
    torch_pin = [TORCH_TRIPLE[0]]
    if run_pip(SPEECH_DEPS + ["kaggle"] + torch_pin + cu128, "speech_deps") != 0:
        print("WARNING: speech deps failed with the torch pin; retrying unpinned.", flush=True)
        if run_pip(SPEECH_DEPS + ["kaggle"], "speech_deps-unpinned") != 0:
            return False

    return True

PYLIBS = find_pylibs([INPUT, WORK])
if PYLIBS:
    _tick(f"deps: warm cache hit -> {PYLIBS}")
    if probe_deps(SHALLOW_PROBE, PYLIBS) != 0:
        print("WARNING: warm pylibs failed its presence check; wiping and reinstalling.", flush=True)
        shutil.rmtree(PYLIBS, ignore_errors=True)
        PYLIBS = None

if PYLIBS is None:
    _tick(f"deps: cold install into {PYLIBS_DIR} (once per cache; ~4-6 min)")
    if install_deps() and probe_deps(DEEP_PROBE, PYLIBS_DIR) == 0:
        PYLIBS = PYLIBS_DIR
    else:
        print("WARNING: pylibs install/probe failed; falling back to a normal system install.", flush=True)
        PYLIBS = None

if PYLIBS:
    # Order matters: pylibs first, so the kernel and every subprocess see the same
    # build of torch that vLLM was resolved against.
    sys.path.insert(0, str(PYLIBS))
    os.environ["PYTHONPATH"] = str(PYLIBS) + os.pathsep + os.environ.get("PYTHONPATH", "")
    try:
        (PYLIBS / READY_MARKER).write_text(
            "installed " + time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()) + "\n", encoding="utf-8"
        )
    except OSError:
        pass  # mounted read-only cache: the marker came with it
    _tick(f"deps: ready from {PYLIBS}")
else:
    # Last resort: the original behaviour. Slower -- pip does uninstall the
    # image's torch here -- but a working run beats a cached one.
    _tick("deps: LAST RESORT installing into system site-packages")
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir"] + SPEECH_DEPS, check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir",
                   "--index-url", "https://download.pytorch.org/whl/cu128", "--no-deps"] + TORCH_TRIPLE,
                   check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir",
                   "vllm==0.26.0"], check=False)

subprocess.run([sys.executable, "-c", "import mazinger; print('Mazinger source import OK')"], check=True)

# Kaggle's newest PyAV release dropped/changed this optional kwarg; faster-whisper still passes it.
# Remove only that compatibility kwarg so audio decode works across PyAV releases.
audio_module = subprocess.check_output(
    [sys.executable, "-c", "import faster_whisper.audio; print(faster_whisper.audio.__file__)"],
    text=True
).strip()
audio_source = Path(audio_module).read_text(encoding="utf-8")
legacy_call = 'av.open(input_file, mode="r", metadata_errors="ignore")'
if legacy_call in audio_source:
    Path(audio_module).write_text(audio_source.replace(legacy_call, 'av.open(input_file, mode="r")'), encoding="utf-8")
    print("Applied PyAV audio decode compatibility patch.", flush=True)
else:
    print("faster-whisper audio decoder has no legacy PyAV call; no patch needed.", flush=True)

try:
    import torch
    gpu_count = torch.cuda.device_count()
except Exception:
    gpu_count = 0
if gpu_count < 1:
    raise RuntimeError("No CUDA GPU is visible. Enable a Kaggle GPU accelerator and rerun.")
tensor_parallel = 1
if gpu_count >= 2:
    print("Using Homura-2B on GPU 0 and reserving GPU 1 for ASR/TTS.", flush=True)
else:
    print("Only one GPU is visible; using Homura-2B on the available GPU.", flush=True)
print(f"Visible CUDA GPUs: {gpu_count}; vLLM tensor parallel size: {tensor_parallel}", flush=True)

# Where the weights come from on THIS run. A mounted/local copy costs 0 bytes;
# otherwise they land in HF_HOME so the next run can reuse them if this output is
# attached as Input. Reported either way: "nothing mounted" is the reason a run is
# still slow, so it must never be silent.
MODEL_PATH = find_model_copy([INPUT, WORK])
if MODEL_PATH:
    _tick(f"model: mounted/local copy at {MODEL_PATH} (no download)")
else:
    _tick(f"model: nothing mounted, downloading into {os.environ['HF_HOME']} (~5 GB, first run only)")

vllm_log = open("/kaggle/working/vllm_server.log", "w", encoding="utf-8")


def vllm_cli():
    """Interpreter for `vllm serve`.

    The console script pip writes into pylibs/bin only imports when PYTHONPATH
    already points at pylibs (set above), so the cached script comes first, then
    whatever a last-resort system install left on PATH.
    """
    if PYLIBS:
        cached = PYLIBS / "bin" / "vllm"
        if cached.exists():
            return [str(cached), "serve"]
    on_path = shutil.which("vllm")
    if on_path:
        return [on_path, "serve"]
    return [sys.executable, "-c", "from vllm.entrypoints.cli.main import main; main()", "serve"]


def start_vllm(model_id):
    server_gpu_count = 1
    server_env = os.environ.copy()
    if gpu_count >= 2:
        # Reserve one T4 for speech models later in the pipeline.
        server_env["CUDA_VISIBLE_DEVICES"] = "0"
    command = vllm_cli() + [
        # Serving a local snapshot path instead of the repo id downloads nothing.
        str(MODEL_PATH) if MODEL_PATH else model_id,
        "--served-model-name", model_id,
        "--host", "127.0.0.1", "--port", "8000",
        "--tensor-parallel-size", str(server_gpu_count),
        "--gpu-memory-utilization", "0.65",
        # --- Task 3: was 4096. The pipeline asks for 8000 output tokens
        # (resegment merge) and 4000+prompt (fit check), and both were rejected
        # with "max_tokens cannot be greater than max_model_len". Homura-2B
        # declares max_position_embeddings=262144, so 16384 is far inside the
        # model and still leaves KV cache headroom on a 16 GB T4.
        "--max-model-len", "16384",
        "--max-num-seqs", "1",
        # Was 2048: raise the per-step token budget so long prompts are not
        # starved now that the context is 4x bigger. Chunked prefill still
        # applies above this, so a bigger prompt degrades instead of failing.
        "--max-num-batched-tokens", "4096",
        "--dtype", "half",
        "--trust-remote-code",
        "--language-model-only",
    ]
    print(f"Starting vLLM model {model_id}: {' '.join(command)}", flush=True)
    return subprocess.Popen(
        command, stdout=vllm_log, stderr=subprocess.STDOUT,
        env=server_env, start_new_session=True
    )


def stop_vllm(process):
    # Kill the whole process group; vLLM may leave CUDA worker children behind.
    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    try:
        process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        pass
    time.sleep(2)
    try:
        os.killpg(process.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass


def wait_for_vllm(process, timeout_seconds):
    deadline = time.monotonic() + timeout_seconds
    while time.monotonic() < deadline:
        if process.poll() is not None:
            return False
        try:
            with urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=5) as response:
                if response.status == 200:
                    return True
        except Exception:
            time.sleep(5)
    return False


# Allow up to 20 minutes for model weight download and server startup.
_vllm_started = time.monotonic()
vllm_process = start_vllm(os.environ["OPENAI_MODEL"])
if not wait_for_vllm(vllm_process, 1200):
    stop_vllm(vllm_process)
    vllm_log.flush()
    tail = Path("/kaggle/working/vllm_server.log").read_text(encoding="utf-8", errors="replace")[-30000:]
    raise RuntimeError("Homura-2B did not start within 20 minutes. vLLM log tail:\n" + tail)
print(f"Local Homura server ready at {os.environ['OPENAI_BASE_URL']} using {os.environ['OPENAI_MODEL']}.", flush=True)
_tick(f"vLLM server up in {time.monotonic() - _vllm_started:.1f}s "
      f"({'no download' if MODEL_PATH else 'weights downloaded'}, "
      f"{'mounted cache' if PYLIBS else 'system install'})")


def du(path):
    """Size of a directory, cheap enough to print on every run."""
    try:
        return subprocess.check_output(["du", "-sh", str(path)], text=True).split()[0]
    except Exception:
        return "?"


print("=== setup timing (target: 60-120 s warm) ===", flush=True)
_tick(f"pylibs={du(PYLIBS) if PYLIBS else 'n/a'}  hf_cache={du(HF_CACHE)}")
_tick(f"TOTAL SETUP {time.monotonic() - T0:.1f}s")


In [ ]:
# =============================================================================
# T_Dubber edge publish -- push the built pylibs tree to the HF repository
# PUBLISH_VARIANT: early
# (injected by edge/inject_cell.py; the early copy exists because a
#  crash after the install must not be able to skip the publish)
# =============================================================================
# WHY THIS CELL EXISTS TWICE
# -------------------------
# This file is injected twice per notebook: the early copy runs right after the
# dependency install, the late copy runs last. inject_cell.py tells them apart by
# the banner line above -- and never by repeating that line in this prose, because
# a marker that appears twice inside one cell cannot be used to find that cell.
#
# Measured reason, from Kaggle run `letest_test3.txt` (2026-10-09): the install
# finished at ~274 s and the tree was complete, but `import torchaudio` raised
# a CUDA mismatch at 545 s, so the kernel stopped -- and a publish cell that only
# runs LAST never ran. The artefact existed and the repository stayed empty, so
# the next cold run would pay the full 458 s pip install again.
#
# Publishing early costs one extra minute of hashing when the run is healthy; the
# late copy then finds the digest already on the Hub and prints "already
# published". A publish attempt is never allowed to fail the run either way.
#
# WHAT THIS CELL IS FOR
# --------------------
# The fetch side (edge/notebook_cell.py) proves the origin works, but a
# repository with nothing published in it saves nothing: the worker prints one
# line and falls back to the 458 s pip install. The artefact can only be built
# by a run that completed its install -- on Kaggle, with Kaggle's linux wheels --
# so THIS run is the only place the tree will ever exist. Publishing at the end
# of the run is what turns the next cold run into a warm one.
#
# WHY PYTHON AND NOT edge-publish.exe
# -----------------------------------
# edge-publish (Go) is the reference publisher and stays byte-compatible with
# edge-fetch, but shipping a second binary through the pack -- and keeping a
# linux build of it current -- costs more than the tar it writes. huggingface_hub
# is already installed for model downloads, and the Hub handles git-lfs for us,
# so this cell needs no binary, no git and no lfs client. It writes the SAME
# layout (manifest.json + artifact/pylibs/pylibs.tar) and the SAME refusal rules
# (.tdubber_ready + vllm/__init__.py), and edge-fetch verifies the digest the
# same way either way.
#
# THE ONE RULE THIS CELL OBEYS
# ----------------------------
# A cache failure is never fatal. No HF_TOKEN secret, no tree, no network, a
# digest that does not match -- each prints one line and exits 0. The run that
# just succeeded must not be failed by the attempt to make the NEXT one faster.
#
# WHY THE UPLOAD ORDER IS TAR FIRST, MANIFEST SECOND
# --------------------------------------------------
# The manifest is what the client trusts. A manifest that names a tar which is
# not on the Hub yet is a window in which every fetch fails verification. The
# tar lands first; the manifest, last. On a digest mismatch against what is
# already published, neither is uploaded at all -- that is the common case, and
# it costs one small GET.
#
# WHY THE MTIME IS FIXED AT ZERO
# ------------------------------
# A re-install of identical wheels produces identical bytes with new mtimes.
# Hashing the file mtimes into the tar would change the digest on every run and
# re-upload gigabytes that did not change. Zeroing mtimes makes the digest a
# function of CONTENT, which is what "already published, skipping" compares.

import argparse
import hashlib
import io
import json
import os
import tarfile
import time
import urllib.error
import urllib.request
from pathlib import Path

DEFAULT_REPO = "pocotarun/tdubber-edge"
DEFAULT_TREE = "/kaggle/working/pylibs"
MANIFEST_REMOTE = "https://huggingface.co/datasets/%s/resolve/main/manifest.json"


def log(msg):
    print("edge-publish-py: %s" % msg, flush=True)


def refuse(msg):
    """The same refusals edge-publish (Go) makes, for the same reason: a tree
    find_pylibs() would ignore still gets downloaded, unpacked and wasted --
    and a half-installed tree published as warm would be worse than no cache
    at all, because the run would skip the install and crash on the import."""
    log("refusing to publish: %s" % msg)
    return 0


def build_tar(tree, out_path):
    """Stream the tree into a deterministic plain .tar and return (size, sha256).

    Plain tar, not tar.gz, is deliberate and matches the Go publisher: the
    payload is wheels and safetensors -- already at the entropy floor -- and a
    measured 400 MB sample of this kind of data compressed to ratio 1.000 while
    costing 74 s of CPU across the 9 GB of a full artefact. Same bytes on the
    wire, a minute and a quarter of CPU saved."""
    h = hashlib.sha256()
    size = 0
    files = sorted(p for p in tree.rglob("*"))
    with open(out_path, "wb") as raw:
        class HashingWriter(io.RawIOBase):
            """A write-only file object that digests every byte through it.

            tarfile calls tell() to track the offset it is writing at, so the
            wrapper counts rather than seeks -- the digest wraps the exact bytes
            that reach the file, tar framing included, which is what edge-fetch
            recomputes after download."""

            def __init__(self):
                self._pos = 0

            def write(self, b):
                nonlocal size
                n = raw.write(b)
                h.update(b[:n])
                size += n
                self._pos += n
                return n

            def tell(self):
                return self._pos

            def writable(self):
                return True

            def seekable(self):
                return False

        with tarfile.open(fileobj=HashingWriter(), mode="w", format=tarfile.GNU_FORMAT) as tf:
            for p in files:
                info = tf.gettarinfo(str(p), arcname="pylibs/" + p.relative_to(tree).as_posix())
                info.uid = info.gid = 0
                info.uname = info.gname = ""
                info.mtime = 0  # content-addressed: see the header note
                if info.isreg():
                    with open(p, "rb") as fh:
                        tf.addfile(info, fh)
                else:
                    tf.addfile(info)
    return size, h.hexdigest()


def read_remote_manifest(repo):
    """The published manifest, or None. Unauthenticated on purpose: the
    repository is public and the check must work exactly as the worker sees it."""
    try:
        with urllib.request.urlopen(MANIFEST_REMOTE % repo, timeout=30) as resp:
            return json.loads(resp.read().decode("utf-8"))
    except (urllib.error.HTTPError, urllib.error.URLError, ValueError):
        return None


def main():
    ap = argparse.ArgumentParser(description="publish the built pylibs tree to the HF origin")
    ap.add_argument("--tree", default=os.environ.get("TDUBBER_EDGE_PUBLISH_TREE", DEFAULT_TREE))
    ap.add_argument("--repo", default=os.environ.get("TDUBBER_EDGE_REPO", DEFAULT_REPO))
    ap.add_argument("--token", default=os.environ.get("HF_TOKEN", ""))
    ap.add_argument("--tar-out", default="", help="keep the tar here instead of beside the tree")
    args = ap.parse_args()

    tree = Path(args.tree)
    if not tree.is_dir():
        log("no tree at %s; nothing to publish (the run may not have reached the install)" % tree)
        return 0
    if not (tree / ".tdubber_ready").is_file():
        log("no .tdubber_ready in %s; refusing (find_pylibs() would ignore this tree)" % tree)
        return 0
    if not (tree / "vllm" / "__init__.py").is_file():
        log("no vllm/__init__.py in %s; refusing (find_pylibs() requires it)" % tree)
        return 0
    if not args.token:
        log("HF_TOKEN is not set (add the HF write token as a notebook secret); skipping")
        return 0

    tar_path = Path(args.tar_out) if args.tar_out else tree.parent / "pylibs.tar"
    log("hashing %s into %s (content-addressed, this takes a minute on a full tree)" % (tree, tar_path))
    started = time.monotonic()
    size, digest = build_tar(tree, tar_path)
    log("tar %d bytes, sha256=%s (%.1f s)" % (size, digest[:16], time.monotonic() - started))

    remote = read_remote_manifest(args.repo)
    for entry in (remote or {}).get("entries", []):
        if entry.get("role") == "pylibs" and entry.get("sha256") == digest and entry.get("size_bytes") == size:
            log("already published (sha256 matches); the repository is current")
            tar_path.unlink(missing_ok=True)
            return 0

    from huggingface_hub import HfApi

    api = HfApi(token=args.token)
    log("uploading artifact/pylibs/pylibs.tar to %s (one-off; git-lfs stores it)" % args.repo)
    started = time.monotonic()
    api.upload_file(
        path_or_fileobj=str(tar_path),
        path_in_repo="artifact/pylibs/pylibs.tar",
        repo_id=args.repo,
        repo_type="dataset",
    )
    log("tar uploaded (%.1f s); writing manifest.json last" % (time.monotonic() - started))

    manifest = {
        "version": 1,
        "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "entries": [
            {
                "name": "pylibs/pylibs.tar",
                "role": "pylibs",
                "size_bytes": size,
                "sha256": digest,
                "modified_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            }
        ],
        "total_bytes": size,
    }
    api.upload_file(
        path_or_fileobj=json.dumps(manifest, indent=2).encode("utf-8") + b"\n",
        path_in_repo="manifest.json",
        repo_id=args.repo,
        repo_type="dataset",
    )
    log("published; the next cold run fetches this instead of pip-installing")
    tar_path.unlink(missing_ok=True)
    return 0


# No sys.exit(): this file is also a notebook cell, and a SystemExit there would
# surface as an error traceback for a run that did not fail. Every path through
# main() returns 0 by design -- a publish failure is a printed line, not an error.
main()


In [ ]:
# Task 4: 10-second pre-flight for the local OpenAI-compatible server.
#
# resegment (max_tokens=8000) and the fit check (4000 + prompt) used to get HTTP
# 400 from a 4096-token server, and BOTH of those call sites swallow the error
# and fall back -- so the notebook only found out after a full GPU run. Prove the
# exact request shape here, before the pipeline: a misconfigured server then
# costs seconds instead of twenty minutes.
import requests

B = "http://127.0.0.1:8000/v1"
print("models:", requests.get(f"{B}/models", timeout=10).status_code)

# 8000 is the largest max_tokens the pipeline asks for (resegment merge). The cap
# is a ceiling, not a target -- "Say OK" stops at EOS -- so this stays fast.
r = requests.post(f"{B}/chat/completions", timeout=90, json={
    "model": "IndexTeam/Index-Homura-2B",
    "messages": [{"role": "user", "content": "Say OK"}],
    "max_tokens": 8000,
})
print("status:", r.status_code)
print(r.text[:300])

if r.status_code != 200:
    raise RuntimeError(
        f"vLLM rejected an 8000-token request (HTTP {r.status_code}). "
        "--max-model-len is still below what the pipeline needs, so resegment's "
        f"merge and the fit check would 400 and silently fall back. Body: {r.text[:400]}"
    )

usage = (r.json() or {}).get("usage", {})
print(
    f"VERIFY PASS: server accepted max_tokens=8000 "
    f"(prompt={usage.get('prompt_tokens')}, completion={usage.get('completion_tokens')})",
    flush=True,
)


In [ ]:
# Task 2: Locate the source video.
#
# P0-3: the host may deliberately ship NO video in this dataset. It leaves the
# original on Telegram instead, so the bytes cross the operator's uplink once
# rather than twice, and this cell restores them with `tgup fetch` (which
# verifies every part digest) before doing anything else.
#
# The mounted dataset video remains the fallback. A worker that cannot reach the
# channel must still be able to start, so nothing here is allowed to turn a
# missing file into a dead kernel on its own.
import os
import glob
import shutil
import json
import sys
from pathlib import Path  # own import: cell 4 must not depend on cell 2

input_dir = "/kaggle/input"
video_path = None
job_config = {"target_language": "Hindi"}
for config_path in Path(input_dir).rglob("dub_job.json"):
    try:
        job_config = json.loads(config_path.read_text(encoding="utf-8"))
        break
    except (OSError, json.JSONDecodeError) as exc:
        print(f"Ignoring unreadable job config {config_path}: {exc}", flush=True)
target_language = str(job_config.get("target_language") or "Hindi")
print(f"Requested target language: {target_language}", flush=True)

if job_config.get("fetch_from_telegram"):
    print("Host left the source on Telegram; restoring it from the channel.", flush=True)
    try:
        mt_hits = sorted(Path(input_dir).rglob("multitasker.py"))
        if not mt_hits:
            raise FileNotFoundError("multitasker.py is not in the dataset")
        sys.path.insert(0, str(mt_hits[0].parent))
        import multitasker as mt
        job = mt.DubJob(
            job_id="job-000",
            video_path="",
            target_language=target_language,
            telegram_backup=str(job_config.get("telegram_backup") or ""),
            source_sha256=str(job_config.get("source_sha256") or ""),
            fetch_from_telegram=True,
            compress_480p=bool(job_config.get("compress_480p_on_worker")),
        )
        workdir = Path("/kaggle/working")
        workdir.mkdir(parents=True, exist_ok=True)
        restored = mt.restore_source_from_channel(job, workdir)
        if restored and Path(restored).is_file() and Path(restored).stat().st_size > 0:
            video_path = restored
            print(f"Restored source at: {video_path}", flush=True)
        else:
            print("Restore did not produce a usable file.", flush=True)
    except Exception as exc:  # noqa: BLE001 - never kill the kernel over this
        print(f"Channel restore failed ({exc}); trying the mounted dataset.", flush=True)

if not video_path:
    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.endswith(('.mp4', '.mkv', '.avi', '.mov')):
                video_path = os.path.join(root, file)
                break
        if video_path:
            break

if not video_path:
    raise FileNotFoundError(
        "No video found: the dataset carried none, the host promised a "
        "telegram_backup link, and the channel restore did not work."
    )
print(f"Found video at: {video_path}")


In [ ]:
# Multitasker (TDUBBER_MULTITASKER=1) -- the pipelined run.
#
# WHY THIS EXISTS
# ---------------
# The single-file cell below dubs one video end to end: ffmpeg,
# then GPU, then upload, each waiting for the previous. On Kaggle
# the GPU and the network are separate hardware, so this cell
# overlaps them with three threads and two bounded queues:
#
#   downloader (ffmpeg staging) --> gpu (mazinger) --> uploader (tgup)
#
# While the GPU dubs job N, the downloader extracts job N+1's
# voice reference and the uploader ships job N-1's finished dub
# to Telegram. With a dub_batch.json in the input dataset the
# whole batch pipelines; with one video, the voice-reference
# extraction still overlaps the first Mazinger stages.
#
# WHERE THE CODE COMES FROM
# -------------------------
# pipeline.py copies multitasker.py (and huggingface/) into the
# input dataset next to source_video.mp4, so the worker imports
# it straight from /kaggle/input -- no pip install, no extra
# dataset, no pack change.
import importlib.util
import json
import os
from pathlib import Path

MULTITASKER_RAN = False
if os.environ.get("TDUBBER_MULTITASKER", "").strip().lower() in ("1", "true", "yes", "on"):
    mt_hits = sorted(Path(input_dir).rglob("multitasker.py"))
    if not mt_hits:
        print("TDUBBER_MULTITASKER=1 but multitasker.py is not in the "
              "input dataset; falling back to the single-file cell below.",
              flush=True)
    else:
        spec = importlib.util.spec_from_file_location("multitasker", mt_hits[0])
        multitasker = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(multitasker)

        jobs = multitasker.discover_jobs(input_dir=Path(input_dir))
        summary = multitasker.run_multitasker(
            jobs=jobs,
            workdir=Path("/kaggle/working"),
            input_dir=Path(input_dir),
            base_url=os.environ.get("OPENAI_BASE_URL", "http://localhost:8000/v1"),
            model_name=os.environ.get("OPENAI_MODEL", ""),
        )
        # Report in the legacy schema so the Kaggle download step
        # and the dashboard keep working unchanged.
        validations = summary.get("validations") or {}
        last_validation = (validations[sorted(validations)[-1]]
                             if validations else {"passed": False, "checks": []})
        report_data = {
            "Status": "Success" if summary["failed"] == 0 else "Failed",
            "Target_Language": target_language,
            "Output_File": "output.mp4" if summary["uploaded"] else "Missing",
            "WER": "Not measured",
            "Sync_Offset": "Not measured",
            "Speaker_Similarity": "Not measured",
            "Multitasker": {k: summary[k] for k in
                              ("jobs", "prepared", "dubbed", "uploaded",
                               "failed", "elapsed_sec")},
            "Validation": last_validation,
        }
        with open("/kaggle/working/report.json", "w", encoding="utf-8") as handle:
            json.dump(report_data, handle, ensure_ascii=False, indent=2)
        if summary["failed"]:
            with open("/kaggle/working/error_log.txt", "w",
                      encoding="utf-8") as handle:
                handle.write(
                    "multitasker: %d/%d job(s) failed; see "
                    "multitasker_ledger.jsonl and per-job "
                    "mazinger_stderr.log\n"
                    % (summary["failed"], summary["jobs"]))
            # Leave a clean runtime: the legacy cell's finally block
            # normally stops vLLM, but it is skipped on this path.
            vllm = globals().get("vllm_process")
            if vllm is not None and vllm.poll() is None:
                vllm.terminate()
            raise RuntimeError(
                "Multitasker run had %d failure(s); see the ledger and "
                "per-job logs under /kaggle/working/multitasker_jobs/."
                % summary["failed"])
        MULTITASKER_RAN = True
        print("Multitasker finished: %s" % summary, flush=True)

In [ ]:
# Lip-sync phase (TDUBBER_LIP_SYNC=1) -- mouth re-render on the freed GPU.
#
# WHY A SEPARATE CELL
# -------------------
# Mazinger dubs the AUDIO; the speaker's mouth still moves in
# the source language. This phase re-renders the mouth region
# of the SOURCE video to match the DUBBED audio: MuseTalk by
# default (latent-space inpainting, ~4 GB VRAM, 30fps+, MIT),
# Wav2Lip as the torch-only fallback when mmcv/mmpose cannot
# build against a newer Kaggle torch.
#
# IT RUNS AFTER vLLM STOPS: the translation LLM and the sync
# model both want the GPU, and a dub never needs them at the
# same time. vLLM is terminated here, GPU 0 goes to the sync
# model, and while the GPU syncs job N the uploader ships
# job N-1's synced video to the Telegram cloud center.
#
# bbox_shift tunes mouth openness (wide-mouth languages such
# as Hindi: try +3..+7). Set TDUBBER_BBOX_SHIFT or put
# "bbox_shift" in dub_job.json / dub_batch.json.
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

LIP_SYNC_RAN = False
if (os.environ.get("TDUBBER_LIP_SYNC", "").strip().lower()
        in ("1", "true", "yes", "on")
        and globals().get("MULTITASKER_RAN", False)):
    eligible = [job for job in jobs if job.lip_sync]
    if not eligible:
        print("TDUBBER_LIP_SYNC=1 but no job is flagged for "
              "lip-sync; skipping the phase.", flush=True)
    else:
        # Free GPU 0 for the sync model (single-GPU boxes).
        vllm = globals().get("vllm_process")
        if vllm is not None and vllm.poll() is None:
            vllm.terminate()
            try:
                vllm.wait(timeout=30)
            except subprocess.TimeoutExpired:
                vllm.kill()
            print("vLLM stopped; GPU 0 handed to the lip-sync model.",
                  flush=True)

        ls_hits = sorted(Path(input_dir).rglob("lip_sync.py"))
        mt_hits = sorted(Path(input_dir).rglob("multitasker.py"))
        if not ls_hits or not mt_hits:
            print("lip_sync.py or multitasker.py missing from the "
                  "input dataset; cannot run the lip-sync phase.",
                  flush=True)
        else:
            spec = importlib.util.spec_from_file_location(
                "lip_sync", ls_hits[0])
            lip_sync = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(lip_sync)

            mt_spec = importlib.util.spec_from_file_location(
                "multitasker", mt_hits[0])
            multitasker = importlib.util.module_from_spec(mt_spec)
            mt_spec.loader.exec_module(multitasker)

            workdir = Path("/kaggle/working")
            provider = lip_sync.build_provider(workdir / "lip_sync")
            phase = multitasker.run_lip_sync_phase(
                jobs, provider, workdir=workdir)
            # The synced video is now the deliverable; refresh the
            # report so the download step and dashboard see it.
            report_path = Path("/kaggle/working/report.json")
            report = {}
            if report_path.is_file():
                report = json.loads(report_path.read_text(
                    encoding="utf-8"))
            report["LipSync"] = phase
            report["Validation"]["passed"] = (
                report.get("Validation", {}).get("passed", False)
                and phase["failed"] == 0)
            report_path.write_text(
                json.dumps(report, ensure_ascii=False, indent=2),
                encoding="utf-8")
            if phase["failed"]:
                raise RuntimeError(
                    "Lip-sync phase had %d failure(s); see "
                    "lip_sync_summary.json and the ledger."
                    % phase["failed"])
            LIP_SYNC_RAN = True
            print("Lip-sync phase finished: %s" % phase, flush=True)

In [ ]:
# Multitasker bypass (TDUBBER_MULTITASKER=1): the pipelined worker
# cell above already ran Mazinger (and optionally the lip-sync
# phase) with download/upload overlap and wrote
# /kaggle/working/output.mp4 + report.json. Nothing left here.
if globals().get("MULTITASKER_RAN", False):
    print("Multitasker already produced the dub -- single-file path skipped.", flush=True)
else:
    # Execute Mazinger against the local Homura vLLM server.
    import traceback
    import subprocess
    import sys

    try:
        print("\U0001f680 Starting Mazinger Dubbing Pipeline...", flush=True)
        base_dir = "/kaggle/working/mazinger_output"
        os.makedirs(base_dir, exist_ok=True)
        if not os.environ.get("OPENAI_MODEL"):
            raise RuntimeError("Local Homura server was not initialized. Rerun the notebook from the first cell.")
        print(f"LLM provider: local {os.environ['OPENAI_MODEL']} via {os.environ['OPENAI_BASE_URL']}.", flush=True)
        voice_sample_path = "/kaggle/working/source_voice_reference.wav"
        subprocess.run([
            "ffmpeg", "-y", "-ss", "6.9", "-i", video_path, "-t", "9.5",
            "-vn", "-ac", "1", "-ar", "24000", "-c:a", "pcm_s16le", voice_sample_path
        ], check=True, capture_output=True)
        cmd = [
            sys.executable, "-m", "mazinger", "dub", video_path,
            "--tts-engine", "omnivoice", "--voice-sample", voice_sample_path,
            "--target-language", target_language,
            "--output-type", "video",
            "--device", "cuda",
            "--transcribe-method", "faster-whisper",
            "--openai-base-url", "http://localhost:8000/v1",
            "--openai-api-key", "EMPTY",
            "--base-dir", base_dir,
            "--llm-model", os.environ["OPENAI_MODEL"]
        ]
        print(f"Executing: {' '.join(cmd)}", flush=True)
        audio_env = os.environ.copy()
        if os.environ["OPENAI_MODEL"].endswith("-2B") and torch.cuda.device_count() >= 2:
            audio_env["CUDA_VISIBLE_DEVICES"] = "1"
        result = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace", env=audio_env)
        print(f"STDOUT:\n{result.stdout}", flush=True)
        print(f"STDERR:\n{result.stderr}", flush=True)
        if result.returncode != 0:
            raise RuntimeError(f"Mazinger CLI failed with return code {result.returncode}")
        print("\u2705 Mazinger finished successfully.", flush=True)

        output_candidates = []
        for root, dirs, files in os.walk(base_dir):
            for file in files:
                if file.lower() == "dubbed.mp4":
                    output_candidates.append(os.path.join(root, file))
        output_candidates = [path for path in output_candidates if os.path.getsize(path) > 0]
        output_video = output_candidates[-1] if output_candidates else None
        if output_video:
            shutil.copy(output_video, "/kaggle/working/output.mp4")
            print("Copied final video to /kaggle/working/output.mp4", flush=True)
        else:
            message = "Could not find a non-empty dubbed.mp4 from Mazinger."
            with open("/kaggle/working/error_log.txt", "w", encoding="utf-8") as error_file:
                error_file.write(message + "\n")
        # A non-empty mp4 alone is not proof of a complete movie dub. Check media
        # duration, required streams, and translated subtitle coverage before
        # reporting success. Feature films are expected to preserve the full source
        # timeline, including quiet scenes and end credits.
        report_data = {
            "Status": "Failed",
            "Target_Language": target_language,
            "Output_File": os.path.basename(output_video) if output_video else "Missing",
            "WER": "Not measured", "Sync_Offset": "Not measured", "Speaker_Similarity": "Not measured",
            "Validation": {"passed": False, "checks": []},
        }
        if output_video:
            def media_info(path):
                result = subprocess.run([
                    "ffprobe", "-v", "error", "-show_entries", "format=duration:stream=codec_type",
                    "-of", "json", path
                ], check=True, capture_output=True, text=True)
                return json.loads(result.stdout)

            source_info = media_info(video_path)
            output_info = media_info(output_video)
            source_duration = float(source_info.get("format", {}).get("duration", 0))
            output_duration = float(output_info.get("format", {}).get("duration", 0))
            stream_types = {stream.get("codec_type") for stream in output_info.get("streams", [])}
            tolerance = max(2.0, source_duration * 0.002)
            duration_ok = source_duration > 0 and output_duration > 0 and abs(source_duration - output_duration) <= tolerance
            streams_ok = "video" in stream_types and "audio" in stream_types

            project_root = Path(base_dir)
            translated_candidates = list(project_root.rglob("translated.raw.srt"))
            if not translated_candidates:
                translated_candidates = list(project_root.rglob("translated.srt"))
            translated_srt = translated_candidates[-1] if translated_candidates else None
            translated_text = translated_srt.read_text(encoding="utf-8", errors="replace") if translated_srt else ""
            subtitle_blocks = sum(1 for line in translated_text.splitlines() if "-->" in line)
            source_candidates = list(project_root.rglob("source.raw.srt"))
            source_text = source_candidates[-1].read_text(encoding="utf-8", errors="replace") if source_candidates else ""
            source_blocks = sum(1 for line in source_text.splitlines() if "-->" in line)
            coverage = subtitle_blocks / source_blocks if source_blocks else 0.0
            # Hindi output should contain Devanagari text in the translated subtitle
            # artifact. This catches the prior case where source-language text was
            # accidentally left unchanged while the job was labeled successful.
            devanagari = sum(1 for char in translated_text if "\u0900" <= char <= "\u097f")
            letters = sum(1 for char in translated_text if char.isalpha())
            script_ratio = devanagari / letters if letters else 0.0
            hindi_script_ok = target_language.lower() != "hindi" or script_ratio >= 0.15
            subtitles_ok = bool(translated_srt and source_blocks > 0 and coverage >= 0.80 and hindi_script_ok)
            checks = [
                {"name": "full_timeline_duration", "passed": duration_ok, "source_seconds": round(source_duration, 2), "output_seconds": round(output_duration, 2), "allowed_drift_seconds": round(tolerance, 2)},
                {"name": "video_and_audio_streams", "passed": streams_ok, "streams": sorted(stream_types)},
                {"name": "translated_subtitle_coverage", "passed": subtitles_ok, "source_segments": source_blocks, "translated_segments": subtitle_blocks, "coverage": round(coverage, 4), "devanagari_ratio": round(script_ratio, 4) if target_language.lower() == "hindi" else None},
            ]
            report_data["Validation"] = {"passed": all(check["passed"] for check in checks), "checks": checks}
            # One line per check: the previous run failed with a single generic
            # RuntimeError and no way to tell duration, streams and subtitle
            # coverage apart without downloading report.json.
            for check in checks:
                print(f"CHECK {check['name']}: {'PASS' if check['passed'] else 'FAIL'} {check}", flush=True)
            report_data["Source_Duration_Seconds"] = round(source_duration, 2)
            report_data["Output_Duration_Seconds"] = round(output_duration, 2)
            if report_data["Validation"]["passed"]:
                report_data["Status"] = "Success"
        with open("/kaggle/working/report.json", "w", encoding="utf-8") as report_file:
            json.dump(report_data, report_file, ensure_ascii=False, indent=2)
        if not output_video:
            raise RuntimeError("Mazinger exited without a non-empty dubbed.mp4 output; inspect error_log.txt.")
        if report_data["Status"] != "Success":
            raise RuntimeError("Output failed full-duration/media/subtitle quality checks. Inspect report.json and artifacts; it will not be reported as a completed dub.")
    except Exception:
        error_text = traceback.format_exc()
        print(f"\u274c Error occurred!\n{error_text}", flush=True)
        with open("/kaggle/working/error_log.txt", "w", encoding="utf-8") as error_file:
            error_file.write(error_text)
        with open("/kaggle/working/report.json", "w", encoding="utf-8") as report_file:
            json.dump({"Status": "Failed", "Target_Language": target_language, "Output_File": "Missing", "WER": "Not measured", "Sync_Offset": "Not measured", "Speaker_Similarity": "Not measured"}, report_file)
        raise
    finally:
        if "vllm_process" in globals() and vllm_process.poll() is None:
            vllm_process.terminate()
            try:
                vllm_process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                vllm_process.kill()
        if "vllm_log" in globals():
            vllm_log.flush()
            vllm_log.close()


In [ ]:
# =============================================================================
# T_Dubber edge publish -- push the built pylibs tree to the HF repository
# PUBLISH_VARIANT: late
# (injected by edge/inject_cell.py; the early copy exists because a
#  crash after the install must not be able to skip the publish)
# =============================================================================
# WHY THIS CELL EXISTS TWICE
# -------------------------
# This file is injected twice per notebook: the early copy runs right after the
# dependency install, the late copy runs last. inject_cell.py tells them apart by
# the banner line above -- and never by repeating that line in this prose, because
# a marker that appears twice inside one cell cannot be used to find that cell.
#
# Measured reason, from Kaggle run `letest_test3.txt` (2026-10-09): the install
# finished at ~274 s and the tree was complete, but `import torchaudio` raised
# a CUDA mismatch at 545 s, so the kernel stopped -- and a publish cell that only
# runs LAST never ran. The artefact existed and the repository stayed empty, so
# the next cold run would pay the full 458 s pip install again.
#
# Publishing early costs one extra minute of hashing when the run is healthy; the
# late copy then finds the digest already on the Hub and prints "already
# published". A publish attempt is never allowed to fail the run either way.
#
# WHAT THIS CELL IS FOR
# --------------------
# The fetch side (edge/notebook_cell.py) proves the origin works, but a
# repository with nothing published in it saves nothing: the worker prints one
# line and falls back to the 458 s pip install. The artefact can only be built
# by a run that completed its install -- on Kaggle, with Kaggle's linux wheels --
# so THIS run is the only place the tree will ever exist. Publishing at the end
# of the run is what turns the next cold run into a warm one.
#
# WHY PYTHON AND NOT edge-publish.exe
# -----------------------------------
# edge-publish (Go) is the reference publisher and stays byte-compatible with
# edge-fetch, but shipping a second binary through the pack -- and keeping a
# linux build of it current -- costs more than the tar it writes. huggingface_hub
# is already installed for model downloads, and the Hub handles git-lfs for us,
# so this cell needs no binary, no git and no lfs client. It writes the SAME
# layout (manifest.json + artifact/pylibs/pylibs.tar) and the SAME refusal rules
# (.tdubber_ready + vllm/__init__.py), and edge-fetch verifies the digest the
# same way either way.
#
# THE ONE RULE THIS CELL OBEYS
# ----------------------------
# A cache failure is never fatal. No HF_TOKEN secret, no tree, no network, a
# digest that does not match -- each prints one line and exits 0. The run that
# just succeeded must not be failed by the attempt to make the NEXT one faster.
#
# WHY THE UPLOAD ORDER IS TAR FIRST, MANIFEST SECOND
# --------------------------------------------------
# The manifest is what the client trusts. A manifest that names a tar which is
# not on the Hub yet is a window in which every fetch fails verification. The
# tar lands first; the manifest, last. On a digest mismatch against what is
# already published, neither is uploaded at all -- that is the common case, and
# it costs one small GET.
#
# WHY THE MTIME IS FIXED AT ZERO
# ------------------------------
# A re-install of identical wheels produces identical bytes with new mtimes.
# Hashing the file mtimes into the tar would change the digest on every run and
# re-upload gigabytes that did not change. Zeroing mtimes makes the digest a
# function of CONTENT, which is what "already published, skipping" compares.

import argparse
import hashlib
import io
import json
import os
import tarfile
import time
import urllib.error
import urllib.request
from pathlib import Path

DEFAULT_REPO = "pocotarun/tdubber-edge"
DEFAULT_TREE = "/kaggle/working/pylibs"
MANIFEST_REMOTE = "https://huggingface.co/datasets/%s/resolve/main/manifest.json"


def log(msg):
    print("edge-publish-py: %s" % msg, flush=True)


def refuse(msg):
    """The same refusals edge-publish (Go) makes, for the same reason: a tree
    find_pylibs() would ignore still gets downloaded, unpacked and wasted --
    and a half-installed tree published as warm would be worse than no cache
    at all, because the run would skip the install and crash on the import."""
    log("refusing to publish: %s" % msg)
    return 0


def build_tar(tree, out_path):
    """Stream the tree into a deterministic plain .tar and return (size, sha256).

    Plain tar, not tar.gz, is deliberate and matches the Go publisher: the
    payload is wheels and safetensors -- already at the entropy floor -- and a
    measured 400 MB sample of this kind of data compressed to ratio 1.000 while
    costing 74 s of CPU across the 9 GB of a full artefact. Same bytes on the
    wire, a minute and a quarter of CPU saved."""
    h = hashlib.sha256()
    size = 0
    files = sorted(p for p in tree.rglob("*"))
    with open(out_path, "wb") as raw:
        class HashingWriter(io.RawIOBase):
            """A write-only file object that digests every byte through it.

            tarfile calls tell() to track the offset it is writing at, so the
            wrapper counts rather than seeks -- the digest wraps the exact bytes
            that reach the file, tar framing included, which is what edge-fetch
            recomputes after download."""

            def __init__(self):
                self._pos = 0

            def write(self, b):
                nonlocal size
                n = raw.write(b)
                h.update(b[:n])
                size += n
                self._pos += n
                return n

            def tell(self):
                return self._pos

            def writable(self):
                return True

            def seekable(self):
                return False

        with tarfile.open(fileobj=HashingWriter(), mode="w", format=tarfile.GNU_FORMAT) as tf:
            for p in files:
                info = tf.gettarinfo(str(p), arcname="pylibs/" + p.relative_to(tree).as_posix())
                info.uid = info.gid = 0
                info.uname = info.gname = ""
                info.mtime = 0  # content-addressed: see the header note
                if info.isreg():
                    with open(p, "rb") as fh:
                        tf.addfile(info, fh)
                else:
                    tf.addfile(info)
    return size, h.hexdigest()


def read_remote_manifest(repo):
    """The published manifest, or None. Unauthenticated on purpose: the
    repository is public and the check must work exactly as the worker sees it."""
    try:
        with urllib.request.urlopen(MANIFEST_REMOTE % repo, timeout=30) as resp:
            return json.loads(resp.read().decode("utf-8"))
    except (urllib.error.HTTPError, urllib.error.URLError, ValueError):
        return None


def main():
    ap = argparse.ArgumentParser(description="publish the built pylibs tree to the HF origin")
    ap.add_argument("--tree", default=os.environ.get("TDUBBER_EDGE_PUBLISH_TREE", DEFAULT_TREE))
    ap.add_argument("--repo", default=os.environ.get("TDUBBER_EDGE_REPO", DEFAULT_REPO))
    ap.add_argument("--token", default=os.environ.get("HF_TOKEN", ""))
    ap.add_argument("--tar-out", default="", help="keep the tar here instead of beside the tree")
    args = ap.parse_args()

    tree = Path(args.tree)
    if not tree.is_dir():
        log("no tree at %s; nothing to publish (the run may not have reached the install)" % tree)
        return 0
    if not (tree / ".tdubber_ready").is_file():
        log("no .tdubber_ready in %s; refusing (find_pylibs() would ignore this tree)" % tree)
        return 0
    if not (tree / "vllm" / "__init__.py").is_file():
        log("no vllm/__init__.py in %s; refusing (find_pylibs() requires it)" % tree)
        return 0
    if not args.token:
        log("HF_TOKEN is not set (add the HF write token as a notebook secret); skipping")
        return 0

    tar_path = Path(args.tar_out) if args.tar_out else tree.parent / "pylibs.tar"
    log("hashing %s into %s (content-addressed, this takes a minute on a full tree)" % (tree, tar_path))
    started = time.monotonic()
    size, digest = build_tar(tree, tar_path)
    log("tar %d bytes, sha256=%s (%.1f s)" % (size, digest[:16], time.monotonic() - started))

    remote = read_remote_manifest(args.repo)
    for entry in (remote or {}).get("entries", []):
        if entry.get("role") == "pylibs" and entry.get("sha256") == digest and entry.get("size_bytes") == size:
            log("already published (sha256 matches); the repository is current")
            tar_path.unlink(missing_ok=True)
            return 0

    from huggingface_hub import HfApi

    api = HfApi(token=args.token)
    log("uploading artifact/pylibs/pylibs.tar to %s (one-off; git-lfs stores it)" % args.repo)
    started = time.monotonic()
    api.upload_file(
        path_or_fileobj=str(tar_path),
        path_in_repo="artifact/pylibs/pylibs.tar",
        repo_id=args.repo,
        repo_type="dataset",
    )
    log("tar uploaded (%.1f s); writing manifest.json last" % (time.monotonic() - started))

    manifest = {
        "version": 1,
        "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "entries": [
            {
                "name": "pylibs/pylibs.tar",
                "role": "pylibs",
                "size_bytes": size,
                "sha256": digest,
                "modified_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            }
        ],
        "total_bytes": size,
    }
    api.upload_file(
        path_or_fileobj=json.dumps(manifest, indent=2).encode("utf-8") + b"\n",
        path_in_repo="manifest.json",
        repo_id=args.repo,
        repo_type="dataset",
    )
    log("published; the next cold run fetches this instead of pip-installing")
    tar_path.unlink(missing_ok=True)
    return 0


# No sys.exit(): this file is also a notebook cell, and a SystemExit there would
# surface as an error traceback for a run that did not fail. Every path through
# main() returns 0 by design -- a publish failure is a printed line, not an error.
main()
